<a href="https://colab.research.google.com/github/zach8421/portable_waste_sorting/blob/main/portable_waste_sorting.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Portable Waste-Sorting Information for Seattle Residents

**IMT 542 — I3 Assignment**

This notebook takes the waste-disposal guidance Seattle Public Utilities publishes through its [Where Does It Go?](https://www.seattle.gov/utilities/your-services/collection-and-disposal/where-does-it-go) tool and reframes it as a *portable* JSON structure that can answer item-level questions ("does this go in recycling, compost, or garbage?") from any interface.

The JSON schema follows the G3 proposal: each item records its name, synonyms, disposal streams, preparation steps, explanation, and a link back to the authoritative source. Applying this structure to real items lets us build a searchable tree, a summary chart, and a simple lookup function on top of the data.

The notebook pulls data directly from two SPU JSON endpoints discovered via browser DevTools:
- `/api/content/taxonomy/3086/pages` — every item (name, synonyms, disposal instructions)
- `/api/content/taxonomy/3087/tree` — the category hierarchy

Items from the first endpoint are transformed into the G3 schema using the category tree from the second, then visualized and queried.


In [1]:
# Install pandas and plotly. Colab has both pre-installed, but this makes the notebook portable.
!pip install -q pandas plotly requests



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [2]:
# Imports. requests hits the API, pandas handles the tabular transforms, plotly draws the tree.
# re, html, and textwrap are stdlib — re/html strip HTML from disposal instructions;
# textwrap wraps long hover-tooltip text so tooltips don't stretch across the screen.
import json
import re
import html
import textwrap
import requests
import pandas as pd
import plotly.graph_objects as go

## 1. Fetch the category tree

The tree endpoint returns the hierarchy of top-level categories (Electronics, Food, Plastic, etc.) and their sub-categories. We flatten it into two lookup dicts — `id_to_name` and `id_to_parent` — so we can turn an item's numeric `Taxonomies` string into a readable category path.


In [3]:
TREE_URL  = "https://seattle.gov/api/content/taxonomy/3087/tree"
PAGES_URL = "https://seattle.gov/api/content/taxonomy/3086/pages"
ROOT_TAXONOMY_ID = "3086"  # the pages root, appears on every item — skipped when building paths

tree = requests.get(TREE_URL, timeout=15).json()

id_to_name, id_to_parent = {}, {}
for top in tree:
    id_to_name[top['id']] = top['name']
    id_to_parent[top['id']] = None
    for child in top.get('children', []):
        id_to_name[child['id']] = child['name']
        id_to_parent[child['id']] = top['id']

print(f"{len(tree)} top-level categories, {len(id_to_name)} total category nodes")
print("Top-level:", ", ".join(top['name'] for top in tree))


14 top-level categories, 56 total category nodes
Top-level: Construction & Demolition, Electronics, Fabric, Food, Food Packaging, Glass & Ceramics, Hazardous Items, Household Items, Metal & Metal Items, Paper, Plastic, Vehicles & Vehicle-Related, Wood, Yard Waste


## 2. Fetch every item and transform to the G3 schema

The pages endpoint returns one record per item. Each record has `Name`, `Alias` (synonyms), `VoiceInstructions` (a plain-language summary), `Taxonomies` (pipe-separated category IDs), and one or more disposal fields (`Garbage`, `Recycling`, `Compost`, `Hazardous`, `Dropoff`, `Special`) containing HTML-formatted instructions. An item with a `Garbage` field *and* a `Hazardous` field has two valid disposal streams — we keep both.

The transform does three things: strip HTML from each disposal field, collect the list of non-empty disposal streams, and build a readable `category_path` from the taxonomy IDs.


In [4]:
# Map each SPU API field to a normalized disposal stream. Multiple API fields
# can share one stream (e.g. both Dropoff and Transferstation mean "take it
# somewhere") — when that happens we concatenate their instructions. The API's
# actual field names (discovered by inspecting responses) don't match the
# resident-facing stream names, so this dict is the translation layer.
FIELD_TO_STREAM = {
    'Recycling':       'recycling',
    'Foodandyard':     'compost',      # API's name for food & yard waste (compost bin)
    'Garbage':         'garbage',
    'Hazardous':       'hazardous',
    'Dropoff':         'dropoff',      # drop at retailer/supplier/special location
    'Transferstation': 'dropoff',      # city transfer station — also a kind of drop-off
    'Curbside':        'special',      # request a scheduled special item pickup
    'Binside':         'special',      # bundle extras next to cart on collection day
    'Donate':          'donate',
    'Reuse':           'donate',       # resell/give-away — grouped with donate
}

def strip_html(text):
    """Convert the SPU HTML snippets to readable plain text."""
    if not text: return ""
    text = html.unescape(text)          # &amp; → &, \u003C → <, etc.
    text = re.sub(r'<li[^>]*>', ' • ', text, flags=re.I)
    text = re.sub(r'<br\s*/?>', ' ', text, flags=re.I)
    text = re.sub(r'<[^>]+>', ' ', text)  # drop remaining tags
    return re.sub(r'\s+', ' ', text).strip()

def build_category_path(taxonomies_str):
    """Turn '3086|3090' into ['Construction & Demolition', 'Floors & Ceiling'].
       Traverses up each leaf ID to build the full breadcrumb."""
    leaves = [tid for tid in taxonomies_str.split('|') if tid != ROOT_TAXONOMY_ID and tid in id_to_name]
    if not leaves:
        return ['Uncategorized']
    leaf = leaves[0]                  # use the first leaf as the primary path
    chain, cur = [], leaf
    while cur is not None:
        chain.insert(0, id_to_name[cur])
        cur = id_to_parent.get(cur)
    return chain

def slugify(name):
    s = re.sub(r'[^a-z0-9]+', '-', name.lower()).strip('-')
    return s or 'unknown'

def to_g3_schema(record):
    """Collapse the API's per-field disposal instructions into a stream-keyed dict."""
    streams = {}
    for field, stream in FIELD_TO_STREAM.items():
        if record.get(field):
            text = strip_html(record[field])
            # If multiple API fields map to the same stream, join their text.
            streams[stream] = streams[stream] + ' ' + text if stream in streams else text
    return {
        'item_id':              record['Id'],
        'item_name':            record['Name'],
        'synonyms':             record.get('Alias', []) or [],
        'disposal_streams':     list(streams.keys()),       # e.g. ['compost', 'dropoff']
        'disposal_conditions':  streams,                    # dict keyed by stream
        'explanation':          record.get('VoiceInstructions', ''),
        'category_path':        build_category_path(record.get('Taxonomies', '')),
        'source_reference':     f"https://www.seattle.gov/utilities/your-services/collection-and-disposal/where-does-it-go#/item/{slugify(record['Name'])}",
        'city_jurisdiction':    'Seattle, WA',
    }

raw_items = requests.get(PAGES_URL, timeout=30).json()
items = [to_g3_schema(r) for r in raw_items]
no_stream = sum(1 for it in items if not it['disposal_streams'])
print(f"Transformed {len(items)} items to the G3 schema ({no_stream} with no disposal stream)")

Transformed 308 items to the G3 schema (1 with no disposal stream)


## 3. Flatten into a DataFrame

Same `pd.json_normalize` pattern as the I3 example notebook's SEC EDGAR cell — one row per item. A `primary_stream` column picks a single disposal category per item for visualizations (using a simple priority: hazardous first, then recycle/compost/garbage, then drop-off/special).


In [5]:
items_df = pd.json_normalize(items)

# When an item has several valid streams (common — e.g. something both
# compostable AND available for drop-off), pick one for colouring and the
# summary bar chart. Order reflects resident priority: hazardous first so those
# items never get visually hidden behind a softer stream classification.
STREAM_PRIORITY = ['hazardous', 'recycling', 'compost', 'garbage', 'dropoff', 'special', 'donate']
def primary_stream(streams):
    for s in STREAM_PRIORITY:
        if s in streams: return s
    return 'other'
items_df['primary_stream'] = items_df['disposal_streams'].apply(primary_stream)

print("Shape:", items_df.shape)
print("\nItems per primary disposal stream:")
print(items_df['primary_stream'].value_counts().to_string())
items_df[['item_name', 'primary_stream', 'disposal_streams', 'category_path']].head(8)

Shape: (308, 16)

Items per primary disposal stream:
primary_stream
garbage      101
recycling     99
hazardous     47
compost       39
dropoff       20
other          1
donate         1


,item_name,primary_stream,disposal_streams,category_path
0,Acetylene Tanks,dropoff,[dropoff],[Uncategorized]
1,Acoustic Ceiling Tile,hazardous,"[garbage, hazardous, dropoff]","[Construction & Demolition, Floors & Ceiling]"
2,Aerosol Cans,hazardous,"[recycling, garbage, hazardous]",[Hazardous Items]
3,"Air Conditioners, Heat Pumps, or Dehumidifiers",garbage,"[garbage, dropoff, special, donate]","[Household Items, Appliances]"
4,Batteries,hazardous,"[hazardous, dropoff, special]","[Household Items, Batteries]"
5,Aluminum & Metal,recycling,"[recycling, garbage, dropoff, special]","[Metal & Metal Items, Metals by Type]"
6,Aluminum Foil,recycling,"[recycling, garbage]","[Metal & Metal Items, From the Kitchen]"
7,Ammunition or Guns,hazardous,[hazardous],[Hazardous Items]


## 4. Visualize the category tree

A Plotly sunburst shows every category, sub-category, and item. Items are coloured by their primary disposal stream so you can see at a glance which categories lean recyclable, which lean hazardous, etc.


In [6]:
# Build sunburst nodes. Every unique (parent, category) pair becomes a node;
# every item becomes a leaf attached to its deepest category.
labels, parents, ids, hover, colours = [], [], [], [], []
CATEGORY_COLOUR = '#dee2e6'
STREAM_COLOUR = {
    'recycling': '#2b8a3e',  # green
    'compost':   '#8a6d1a',  # olive/gold — food & yard waste
    'garbage':   '#495057',  # dark grey
    'hazardous': '#c92a2a',  # red
    'dropoff':   '#1864ab',  # blue
    'special':   '#6741d9',  # purple
    'donate':    '#0c8599',  # teal
    'other':     '#868e96',  # medium grey — items with no stream info in the API
}
# Pretty legend labels for each normalized stream key.
STREAM_DISPLAY = {
    'recycling': 'Recycling',
    'compost':   'Compost (food & yard)',
    'garbage':   'Garbage',
    'hazardous': 'Hazardous',
    'dropoff':   'Drop-off',
    'special':   'Special pickup',
    'donate':    'Donate / reuse',
    'other':     'Other',
}

seen_cats = set()
def add_category_chain(chain):
    parent_id = ''
    for name in chain:
        node_id = f"{parent_id}/{name}" if parent_id else name
        if node_id not in seen_cats:
            seen_cats.add(node_id)
            ids.append(node_id); labels.append(name)
            parents.append(parent_id); hover.append(f"Category: {name}")
            colours.append(CATEGORY_COLOUR)
        parent_id = node_id
    return parent_id  # deepest category id

for item in items:
    leaf_parent = add_category_chain(item['category_path'])
    ids.append(item['item_id']); labels.append(item['item_name'])
    parents.append(leaf_parent)
    stream = primary_stream(item['disposal_streams'])
    # Wrap explanation at ~80 chars so the hover tooltip reads vertically
    # instead of stretching off-screen. textwrap.wrap splits on word boundaries.
    explanation_wrapped = "<br>".join(textwrap.wrap(item['explanation'], width=80)) if item['explanation'] else ""
    # Show the pretty stream names in the tooltip too, matching the legend.
    stream_labels = ", ".join(STREAM_DISPLAY.get(s, s) for s in item['disposal_streams']) or 'none'
    hover.append(
        f"<b>{item['item_name']}</b><br>"
        f"Streams: {stream_labels}<br>"
        f"{explanation_wrapped}"
    )
    colours.append(STREAM_COLOUR.get(stream, STREAM_COLOUR['other']))

fig = go.Figure()
fig.add_trace(go.Sunburst(
    labels=labels, parents=parents, ids=ids,
    hovertext=hover, hoverinfo='text',
    marker=dict(colors=colours), branchvalues='total',
))

# Sunburst has no native legend (the slices ARE the data). We add invisible
# Scatter traces with x=[None], y=[None] — nothing draws on the canvas, but
# each one creates a colored entry in the legend on the right. Order and
# labels here come straight from STREAM_DISPLAY so they cannot drift from
# the colours used on the chart.
LEGEND_ORDER = ['recycling', 'compost', 'garbage', 'hazardous', 'dropoff', 'special', 'donate', 'other']
for stream in LEGEND_ORDER:
    fig.add_trace(go.Scatter(
        x=[None], y=[None], mode='markers',
        marker=dict(size=14, color=STREAM_COLOUR[stream], symbol='square'),
        name=STREAM_DISPLAY[stream], showlegend=True,
    ))
# Plus an entry for the grey rings that represent the category hierarchy.
fig.add_trace(go.Scatter(
    x=[None], y=[None], mode='markers',
    marker=dict(size=14, color=CATEGORY_COLOUR, symbol='square',
                line=dict(color='#adb5bd', width=1)),
    name='Category', showlegend=True,
))

fig.update_layout(
    title=f"Seattle Waste Disposal — {len(items)} items across {len(tree)} categories",
    width=1150, height=800, margin=dict(t=60, l=10, r=230, b=10),
    xaxis=dict(visible=False), yaxis=dict(visible=False),
    showlegend=True,
    legend=dict(
        title="<b>Disposal stream</b>",
        x=1.02, y=0.5, xanchor='left', yanchor='middle',
        bgcolor='rgba(255,255,255,0.9)', bordercolor='#dee2e6', borderwidth=1,
    ),
    hoverlabel=dict(align='left', bgcolor='white', bordercolor='#343a40', font=dict(size=13)),
)
fig.show()

## 5. Distribution across disposal streams

Quick bar chart of items grouped by primary disposal stream. This tells you which streams dominate Seattle's guidance — useful if you're trying to figure out which category to memorize first.


In [7]:
counts = items_df['primary_stream'].value_counts().reset_index()
counts.columns = ['stream', 'count']

bar = go.Figure(go.Bar(
    x=counts['stream'], y=counts['count'],
    marker_color=[STREAM_COLOUR.get(s, STREAM_COLOUR['other']) for s in counts['stream']],
    text=counts['count'], textposition='outside',
))
bar.update_layout(
    title="Items per primary disposal stream",
    xaxis_title="Disposal stream", yaxis_title="Number of items",
    width=800, height=450, plot_bgcolor='#fafafa',
)
bar.show()


## 6. Item lookup — the portable use case

The point of expressing this data in the G3 schema is that the *same* JSON can power any interface. Here's a minimal Python lookup that answers "where does this go?" using item names or synonyms. Swap the body for JavaScript, Swift, or a Flask endpoint and the structure still holds.


In [8]:
def lookup(query, items=items, limit=5):
    """Find matching items by name or synonym."""
    q = query.lower().strip()
    hits = []
    for item in items:
        haystack = [item['item_name'].lower()] + [s.lower() for s in item['synonyms']]
        if any(q == h or q in h for h in haystack):
            hits.append(item)
    return hits[:limit]

def explain(query):
    results = lookup(query)
    if not results:
        print(f"No match for '{query}'.")
        return
    for r in results:
        print(f"\n→ {r['item_name'].upper()}")
        print(f"   Category:  {' > '.join(r['category_path'])}")
        print(f"   Streams:   {', '.join(r['disposal_streams']) or '(none listed)'}")
        if r['explanation']:
            print(f"   Summary:   {r['explanation'][:300]}")
        for stream, text in r['disposal_conditions'].items():
            print(f"   — {stream.title()}: {text[:250]}")
        print(f"   Source:    {r['source_reference']}")

explain("batteries")
explain("pizza boxes")
explain("plastic bags")



→ BATTERIES
   Category:  Household Items > Batteries
   Streams:   hazardous, dropoff, special
   Summary:   Do not put batteries in your curbside trash or recycling bins. This includes Alkaline, Lithium, Lithium-Ion, Rechargeable, Computer, and Button batteries as well as products that contain batteries that cannot be easily removed. It is important to keep batteries out of your bins because they can caus
   — Hazardous: Batteries can't go in the garbage because they contain materials harmful to people, animals, and the environment.
   — Dropoff: • Bring batteries to a household hazardous waste facility or city transfer station , where the materials will be reclaimed and recycled. No fee. • Accepted batteries include alkaline, button, rechargeable, and lead-acid batteries. • Separate batterie
   — Special: Submit a request online or call (206) 684-3000 to schedule a pickup from your residence. The materials collected will be reclaimed and recycled. Fees apply.
   Source:    https://

## 7. Save a snapshot

Writing the transformed items and the tree to `data/items.json` gives us a versioned snapshot that can be loaded offline or used by other applications without hitting the live API. The file is the portable deliverable; the notebook is just one consumer of it.


In [9]:
snapshot = {
    'metadata': {
        'source': 'Seattle Public Utilities — Where Does It Go?',
        'source_urls': [TREE_URL, PAGES_URL],
        'city_jurisdiction': 'Seattle, WA',
        'schema_version': '1.0',
        'item_count': len(items),
        'generated_at': pd.Timestamp.utcnow().isoformat(),
    },
    'category_tree': tree,
    'items': items,
}

import os
os.makedirs('data', exist_ok=True)
with open('data/items.json', 'w') as f:
    json.dump(snapshot, f, indent=2, ensure_ascii=False)

size_kb = os.path.getsize('data/items.json') / 1024
print(f"Saved data/items.json — {len(items)} items, {size_kb:.1f} KB")


Saved data/items.json — 308 items, 384.3 KB


## Summary

- Two SPU JSON endpoints (pages + tree) are all that's needed to get the full dataset — no scraping, no headless browser.
- Each record transforms to the G3 schema in about 20 lines of code, HTML-stripping included.
- The sunburst surfaces patterns (which categories lean hazardous vs. recyclable) that aren't obvious from the A-Z list.
- A 15-line `lookup()` function is a full working implementation of the portable "where does this go?" use case.
